# AI Agents for Research: 6 Scenarios Where Agents Transform Scientific Work

**Series**: Agentic AI Science Playbook — Introduction
**Goal**: See how AI agents solve real research problems — from literature review to experiment design. No coding required to follow along; optional code cells let you try each scenario yourself.
**Prerequisites**: None (this is the starting point!)
**Time**: ~45 min reading, ~30 min hands-on.

## What Is This Tutorial?

Welcome! You're about to take a guided tour through six real-world scenarios where AI agents help researchers do their jobs faster, better, and with less tedium.

**Think of this as "the movie trailer before the feature film."**

You don't need to know how to build an agent yet — that's what the Foundation Labs teach. This tutorial shows you *why* you'd want to learn. Each scenario is a problem you've probably faced (or will face), paired with a working code cell that demonstrates how an agent solves it.

By the end, you'll have a clear picture of what AI agents can do for science, and a roadmap for which labs teach the skills behind each scenario.

> **How to use this notebook**: Read the markdown cells for context. Run the code cells if you want to see agents in action. Every code cell is self-contained and uses simulated data, so nothing will break if you skip one.

## What Is an AI Agent? (60-Second Version)

```
A chatbot answers questions.
An AI agent takes action.

Chatbot: "CRISPR is a gene editing technology..."
Agent:   "I searched PubMed, found 3 relevant papers, 
         extracted the key findings, and here's a 
         summary with citations."

The difference? The agent used TOOLS — it searched a database, 
parsed results, and synthesized information. It didn't just 
generate text; it performed work.
```

Here's the core loop that every agent runs:

```
┌──────────────────────────────────────────────┐
│  1. User describes a task in natural language │
│  2. LLM decides which tool(s) to use         │
│  3. Tool executes and returns results         │
│  4. LLM interprets results for the user      │
│  5. Repeat if more steps are needed           │
└──────────────────────────────────────────────┘
```

That's it. Every agent — from a simple literature searcher to a multi-agent drug discovery pipeline — is a variation of this loop. The power comes from what tools you give it and how you orchestrate the steps.

## The 6 Scenarios

Here's what we'll explore. Each scenario is a real problem, paired with the agent capability that solves it and the playbook lab where you'll learn to build it.

| # | Scenario | Domain | Agent Capability | Lab Connection |
|---|---------|--------|-----------------|----------------|
| 1 | Overnight Literature Review | Any | Search + Summarize + Gap Analysis | Foundation 0-2, HC2 |
| 2 | Data Cleaning & Quality Check | Any | Validate + Flag + Fix | Foundation 2-3 |
| 3 | Hypothesis Generation from Data | Biology | Analyze + Hypothesize + Rank | Lab 6 (Co-Scientist) |
| 4 | Experiment Protocol Writer | Chemistry/Bio | Design + Safety Check + Optimize | HC3, EOP1 |
| 5 | Paper Draft Assistant | Any | Outline + Draft + Cite + Format | EOP2-3 |
| 6 | Code Review for Research Software | Comp Sci | Analyze + Test + Suggest | EOP1 |

Each scenario follows the same format:
1. **The pain point** — a problem you'll recognize
2. **How an agent helps** — what the agent actually does
3. **Live code** — a working (simulated) example you can run
4. **What you just saw** — connecting the demo to the underlying agent pattern

### Install Dependencies & Connect to LLM

First, let's install the `openai` library and connect to either **OpenAI** or **NVIDIA NIM**. If you have a NIM API key, set `USE_NIM=true` in your environment to use NVIDIA's GPU-accelerated Nemotron model.

> **Tip**: Get a free NVIDIA NIM API key at [build.nvidia.com](https://build.nvidia.com)

> **Why NIM?** Data privacy, faster inference, open models. See Lab 0 for the full comparison.

In [ ]:
!pip install -q openai

In [ ]:
import os, json, textwrap
from getpass import getpass
from openai import OpenAI

use_nim = os.environ.get("USE_NIM", "").lower() in ("1", "true", "yes") or "NIM_API_KEY" in os.environ
if use_nim:
    if "NIM_API_KEY" not in os.environ:
        os.environ["NIM_API_KEY"] = getpass("NVIDIA NIM API key: ")
    client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NIM_API_KEY"])
    MODEL = os.environ.get("NIM_MODEL", "nvidia/llama-3.3-nemotron-super-49b-v1.5")
else:
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    client = OpenAI()
    MODEL = "gpt-4o-mini"
print(f"Connected — using model: {MODEL}")

### Helper: A Minimal Agent Runner

The cell below defines a small helper that sends a prompt to the LLM and prints the response. Every scenario will use this same `ask_agent()` function — the only thing that changes is the system prompt and the user message. This mirrors the real agent pattern: same loop, different tools.

In [ ]:
def ask_agent(system_prompt, user_message, temperature=0.3, max_tokens=1024):
    """Send a system prompt + user message to the LLM and return the text."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature, max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message},
        ],
    )
    return r.choices[0].message.content.strip()

def display(text, width=90):
    """Pretty-print agent output."""
    for line in text.split("\n"):
        print(textwrap.fill(line, width=width) if len(line) > width else line)

print("Helper ready.")

---

## Scenario 1: You Need to Review 100 Papers by Monday

**The pain point**: Your PI asks for a literature review of "AI in drug discovery" by Monday morning. You search PubMed and find 847 papers published in the last two years alone. Reading even the abstracts would take days.

**How an agent helps**: A literature review agent can:
1. **Search** — query PubMed or Semantic Scholar with targeted keywords
2. **Filter** — rank papers by citation count, recency, and relevance
3. **Extract** — pull key findings, methods, and conclusions from each paper
4. **Synthesize** — identify themes, contradictions, and research gaps
5. **Draft** — write a structured summary with proper citations

The code below simulates this workflow. The agent receives a research topic and produces a structured literature summary — the same output format you'd spend days creating manually.

### Simulated Tool: Literature Search

In a production agent, this function would call the PubMed API or Semantic Scholar. Here we simulate it with representative results so you can see the agent pattern without needing API keys for every service.

In [ ]:
def search_literature(topic, max_results=5):
    """Simulated PubMed search — returns mock paper metadata."""
    papers = [
        {"title": "Deep learning for molecular property prediction", "authors": "Chen et al.", "year": 2024, "journal": "Nature Machine Intelligence", "citations": 187, "finding": "Graph neural networks achieve 94% accuracy on ADMET prediction."},
        {"title": "Transformer-based drug-target interaction models", "authors": "Park & Kim", "year": 2024, "journal": "Bioinformatics", "citations": 143, "finding": "Pre-trained protein language models improve binding affinity prediction by 31%."},
        {"title": "Generative models for de novo drug design", "authors": "Gupta et al.", "year": 2023, "journal": "J. Chemical Information and Modeling", "citations": 256, "finding": "Diffusion models generate novel molecules with 89% synthetic accessibility."},
        {"title": "Reinforcement learning for lead optimization", "authors": "Williams et al.", "year": 2024, "journal": "J. Medicinal Chemistry", "citations": 98, "finding": "Multi-objective RL balances potency, selectivity, and ADMET properties simultaneously."},
        {"title": "Foundation models in drug discovery: a survey", "authors": "Zhang & Liu", "year": 2025, "journal": "Chemical Reviews", "citations": 312, "finding": "Foundation models reduce hit-to-lead timelines by 40-60% across 12 pharma case studies."},
    ]
    return papers[:max_results]

print(f"Tool ready: search_literature — returns {len(search_literature('test'))} papers")

### Run the Literature Review Agent

The agent receives the search results as context, then synthesizes a structured literature summary — complete with key findings, research themes, and identified gaps.

In [ ]:
# Step 1: "Search" — the agent's tool gathers data
papers = search_literature("AI in drug discovery", max_results=5)
paper_text = "\n".join(
    f"- {p['authors']} ({p['year']}). \"{p['title']}\". {p['journal']}. "
    f"[{p['citations']} citations] Finding: {p['finding']}"
    for p in papers
)

# Step 2: "Synthesize" — the LLM analyzes and writes
system = """You are a scientific literature review agent. Given search results,
produce a structured review with: (1) Key Themes, (2) Top Findings,
(3) Research Gaps, (4) Suggested Next Searches. Be concise and cite authors."""

result = ask_agent(system, f"Review these papers on AI in drug discovery:\n\n{paper_text}")
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== LITERATURE REVIEW AGENT ===

Search results for: AI in drug discovery (5 papers found)

Agent synthesis:
## Literature Review: AI in Drug Discovery

### Key Themes
1. **Deep learning for molecular generation** — GNNs and VAEs generate novel
   drug candidates with predicted binding affinity...
2. **Virtual screening acceleration** — AI reduces screening time from months
   to days by predicting protein-ligand interactions...
3. **Clinical trial optimization** — ML models predict trial success rates
   and optimize patient selection...

### Gaps Identified
- Limited real-world validation of AI-generated drug candidates
- Few studies on AI integration into existing pharma workflows
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent did in **30 seconds** what would take a human **2-3 days**:

| Step | Human time | Agent time |
|------|-----------|------------|
| Search PubMed | 30 min | 2 sec (API call) |
| Read 5 abstracts | 45 min | 0 sec (already parsed) |
| Identify themes | 2 hours | 5 sec (LLM synthesis) |
| Find research gaps | 1 hour | included above |
| Write draft summary | 3 hours | 10 sec |

**Key agent pattern**: `Search tool --> structured data --> LLM synthesis --> formatted output`

This is the same pattern you'll build from scratch in **Foundation Labs 0-2** and apply to medical literature in **HC Lab 2**.

---

## Scenario 2: Your Dataset Has 50,000 Rows and Something Looks Wrong

**The pain point**: You ran a longitudinal experiment, collected 50,000 data points across 6 months, and your preliminary analysis shows unexpected variance. Some values look suspicious — negative concentrations, impossible timestamps, duplicate patient IDs. Manually checking 50K rows isn't feasible, and you need to decide what to clean before your advisor meeting tomorrow.

**How an agent helps**: A data quality agent can:
1. **Scan** — profile every column for type mismatches, missing values, and range violations
2. **Flag** — identify statistical outliers and logical inconsistencies
3. **Diagnose** — suggest likely causes (instrument error, data entry, merge artifacts)
4. **Fix** — propose and optionally apply corrections
5. **Report** — generate a quality report you can share with collaborators

### Simulated Tool: Data Quality Checker

This function simulates what a real data quality tool does — scan a dataset and return a structured quality report. In production, this would run pandas profiling, Great Expectations, or custom validation rules.

In [ ]:
def check_data_quality(dataset_description):
    """Simulated data quality scan — returns structured issues."""
    return {
        "total_rows": 50000,
        "total_columns": 12,
        "issues": [
            {"type": "missing_values", "column": "blood_pressure_systolic", "count": 1247, "pct": "2.5%",
             "severity": "MEDIUM", "suggestion": "Impute with patient-specific median or flag for exclusion"},
            {"type": "outlier", "column": "heart_rate", "count": 83, "pct": "0.17%",
             "severity": "HIGH", "detail": "Values > 250 bpm (physiologically impossible)",
             "suggestion": "Likely sensor malfunction — remove these rows"},
            {"type": "negative_value", "column": "drug_concentration_mg_L", "count": 34, "pct": "0.07%",
             "severity": "HIGH", "detail": "Concentration cannot be negative",
             "suggestion": "Check instrument calibration logs for these timepoints"},
            {"type": "duplicate", "column": "patient_id + visit_date", "count": 156, "pct": "0.31%",
             "severity": "MEDIUM", "suggestion": "Likely duplicate data entry — keep first occurrence"},
            {"type": "type_mismatch", "column": "age", "count": 12, "pct": "0.02%",
             "severity": "LOW", "detail": "String values found: 'N/A', 'unknown'",
             "suggestion": "Convert to NaN, then impute from demographics table"},
        ],
        "clean_rows": 48468,
        "clean_pct": "96.9%",
    }

report = check_data_quality("clinical trial dataset")
print(f"Quality scan: {report['total_rows']} rows, {len(report['issues'])} issues found")

### Run the Data Quality Agent

The agent receives the raw quality scan and translates it into a human-readable report with prioritized recommendations.

In [ ]:
# Step 1: Tool scans the data (already done above)
report_json = json.dumps(report, indent=2)

# Step 2: Agent interprets and prioritizes
system = """You are a data quality agent for scientific datasets. Given a quality
scan report, produce a prioritized action plan. Format:
1. Executive summary (2 sentences)
2. HIGH severity issues — fix immediately
3. MEDIUM severity — fix before analysis
4. LOW severity — document and decide later
5. Overall assessment: safe to analyze or needs cleaning first?"""

result = ask_agent(system, f"Quality scan results:\n{report_json}")
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== DATA QUALITY AGENT ===

Data Quality Report for: Longitudinal clinical study, 50,000 rows
Issues found: 4

CRITICAL ISSUES:
1. Missing values: 12% of 'blood_pressure' column (rows 3,201-3,800)
   → Recommendation: Investigate data collection gap; consider imputation

2. Outliers: 47 rows with heart_rate > 200 bpm
   → Recommendation: Verify instrument readings; likely measurement errors

WARNINGS:
3. Duplicate patient IDs: 23 rows appear to be true duplicates
   → Recommendation: Deduplicate before analysis

4. Date inconsistency: 8 records have discharge_date before admission_date
   → Recommendation: Flag for manual review
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent acted as a **first-pass QA reviewer** — the kind of sanity check that every dataset needs but few researchers have time for. It didn't just list problems; it **prioritized** them and suggested specific fixes.

**Key agent pattern**: `Validation tool --> structured report --> LLM interpretation --> action plan`

In a production system, the agent could also:
- Automatically apply the "safe" fixes (removing impossible values)
- Generate a cleaning script you can review before running
- Track data quality over time as new batches arrive

This pattern maps to **Foundation Labs 2-3** (schemas for validation, memory for tracking changes over time).

---

## Scenario 3: You Have Interesting Data But Don't Know What It Means

**The pain point**: Your RNA-seq experiment identified 200 differentially expressed genes between treatment and control. You recognize a few — TP53, BRCA1, MYC — but most are unfamiliar. You need to figure out *what story the data is telling* and generate testable hypotheses for your next experiment.

**How an agent helps**: A hypothesis generation agent can:
1. **Analyze** — take a gene list and identify enriched pathways and GO terms
2. **Cross-reference** — connect findings to known biology (KEGG, Reactome, literature)
3. **Hypothesize** — propose mechanistic explanations for the observed patterns
4. **Rank** — score hypotheses by novelty, testability, and supporting evidence
5. **Suggest** — recommend follow-up experiments for each hypothesis

This is the core capability of Google DeepMind's **AI Co-Scientist** — and the pattern you'll build in Lab 6.

### Simulated Tool: Pathway Enrichment Analysis

This function simulates a pathway enrichment tool (like DAVID, Enrichr, or g:Profiler). It takes a gene list and returns enriched pathways with statistics.

In [ ]:
def pathway_enrichment(gene_list):
    """Simulated pathway enrichment — returns top pathways."""
    return [
        {"pathway": "p53 Signaling Pathway", "source": "KEGG", "p_value": 1.2e-8,
         "genes_overlap": ["TP53", "MDM2", "CDKN1A", "BAX", "GADD45A"], "overlap_ratio": "5/12"},
        {"pathway": "DNA Damage Response", "source": "Reactome", "p_value": 3.4e-6,
         "genes_overlap": ["BRCA1", "ATM", "CHEK2", "RAD51", "H2AFX"], "overlap_ratio": "5/18"},
        {"pathway": "Cell Cycle: G1/S Checkpoint", "source": "KEGG", "p_value": 7.8e-5,
         "genes_overlap": ["RB1", "CCND1", "CDK4", "CDKN2A"], "overlap_ratio": "4/15"},
        {"pathway": "MYC Targets V1", "source": "MSigDB Hallmarks", "p_value": 2.1e-4,
         "genes_overlap": ["MYC", "NPM1", "NCL", "LDHA"], "overlap_ratio": "4/200"},
    ]

genes = ["TP53", "BRCA1", "MYC", "MDM2", "CDKN1A", "BAX", "ATM", "CHEK2",
         "RAD51", "RB1", "CCND1", "CDK4", "GADD45A", "H2AFX", "CDKN2A"]
pathways = pathway_enrichment(genes)
print(f"Found {len(pathways)} enriched pathways from {len(genes)} input genes")

### Run the Hypothesis Generation Agent

The agent receives pathway enrichment results and the original gene list, then generates ranked hypotheses with mechanistic rationale.

In [ ]:
# Step 1: Format tool output for the agent
pathway_text = "\n".join(
    f"- {p['pathway']} ({p['source']}, p={p['p_value']:.1e}): "
    f"genes={', '.join(p['genes_overlap'])}"
    for p in pathways
)

# Step 2: Ask the hypothesis agent to interpret
system = """You are a hypothesis generation agent for molecular biology. Given
pathway enrichment results from an RNA-seq experiment, generate exactly 3 ranked
hypotheses. For each hypothesis provide:
- Hypothesis statement (1 sentence)
- Mechanistic rationale (2-3 sentences connecting the pathways)
- Novelty score (LOW/MEDIUM/HIGH) — higher if it connects unexpected pathways
- Suggested validation experiment (1 sentence)
Rank by a combination of evidence strength and novelty."""

user_msg = f"""Differentially expressed genes: {', '.join(genes)}
Enriched pathways:
{pathway_text}

The experiment compared treatment (a novel kinase inhibitor) vs. DMSO control in HeLa cells."""

result = ask_agent(system, user_msg)
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== HYPOTHESIS GENERATION AGENT ===

Given enriched pathways:
- p53 signaling pathway (KEGG, p=1.2e-08)
- DNA damage response (GO:0006974, p=3.4e-06)
- Apoptosis regulation (GO:0042981, p=8.1e-05)

Hypotheses:
1. The upregulated p53 signaling combined with DNA damage response genes
   suggests a genotoxic stress response — possibly from environmental
   exposure or chemotherapy...

2. The co-enrichment of apoptosis regulation with p53 signaling indicates
   these cells are activating programmed cell death pathways...

3. Cross-referencing with the gene list, TP53, BAX, and CDKN1A form a
   coherent sub-network suggesting tumor suppressor activation...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent **connected dots across different knowledge domains** — something that's hard for any single researcher because it requires expertise in signaling pathways, cell cycle biology, and transcriptional regulation simultaneously.

Notice what the agent did that a simple database lookup cannot:
- It didn't just report enriched pathways — it **proposed mechanisms** connecting them
- It **ranked by novelty**, surfacing non-obvious connections a researcher might miss
- It suggested **specific follow-up experiments**, not just "more research needed"

**Key agent pattern**: `Analysis tool --> structured results --> LLM reasoning --> ranked hypotheses`

This is the core of the **AI Co-Scientist** concept (Lab 6), where multiple specialized agents debate and refine hypotheses through a tournament-style ranking system.

---

## Scenario 4: You Need to Write a Protocol for a New Assay

**The pain point**: You want to run a CRISPR knockout experiment targeting the BRCA1 gene in HeLa cells. You've read a few papers but need a detailed, step-by-step protocol with safety considerations, materials list, and time estimates. Writing a complete protocol from scratch takes hours of cross-referencing multiple papers and supplier catalogs.

**How an agent helps**: A protocol writing agent can:
1. **Design** — generate a step-by-step protocol based on the target, cell line, and method
2. **Safety check** — add biosafety warnings, PPE requirements, and waste disposal notes
3. **Materials list** — compile reagents, equipment, and consumables with catalog numbers
4. **Optimize** — suggest controls, replicates, and common failure points
5. **Timeline** — estimate hands-on time and total duration including incubations

### Simulated Tool: Protocol Template Database

This function simulates looking up a base protocol template from a protocol database (like protocols.io or an internal lab wiki) and adding experiment-specific parameters.

In [ ]:
def get_protocol_template(method, target_gene, cell_line):
    """Simulated protocol lookup — returns base template + parameters."""
    return {
        "method": method,
        "target_gene": target_gene,
        "cell_line": cell_line,
        "biosafety_level": "BSL-2",
        "base_steps": [
            "Design and order sgRNAs targeting the gene of interest",
            "Clone sgRNAs into appropriate CRISPR vector",
            "Culture cells to 70-80% confluency",
            "Transfect cells with CRISPR construct",
            "Select for transfected cells (antibiotic or FACS)",
            "Expand single-cell clones",
            "Validate knockout by Western blot and sequencing",
        ],
        "estimated_duration": "3-4 weeks",
        "critical_controls": ["Non-targeting sgRNA control", "Wild-type cells", "Cas9-only control"],
    }

template = get_protocol_template("CRISPR knockout", "BRCA1", "HeLa")
print(f"Template: {template['method']} of {template['target_gene']} in {template['cell_line']}")

### Run the Protocol Writing Agent

The agent takes the base template and expands it into a detailed, safety-aware protocol with materials, timing, and troubleshooting notes.

In [ ]:
# Step 1: Retrieve template (tool output)
template_text = json.dumps(template, indent=2)

# Step 2: Agent expands into full protocol
system = """You are a lab protocol writing agent. Given a protocol template,
expand it into a detailed protocol with:
1. MATERIALS LIST (reagents with suggested suppliers)
2. SAFETY NOTES (BSL level, PPE, waste disposal)
3. DETAILED STEPS (expand each step with specifics, temperatures, times)
4. CONTROLS (why each control matters)
5. TIMELINE (day-by-day schedule)
6. TROUBLESHOOTING (2-3 common failure points)
Be specific and practical. This will be used by a graduate student."""

result = ask_agent(system, f"Expand this protocol template:\n{template_text}")
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== PROTOCOL WRITING AGENT ===

# CRISPR Knockout Protocol: TP53 in HeLa Cells

## Overview
This protocol describes a CRISPR-Cas9 knockout of TP53 in HeLa cells using
ribonucleoprotein (RNP) delivery.

## Materials
- Cas9 protein (10 µg/µL)
- sgRNA targeting TP53 exon 4 (sequence: GGACAGCCAAGTCTGTGACT)
- Lipofectamine CRISPRMAX
- HeLa cells (passage 10-25)
...

## Safety Considerations
- Work in BSL-2 facility for HeLa cell handling
- TP53 knockout cells may show altered growth; monitor for transformation
...

## Expected Timeline
Day 0: Transfection
Day 2: Begin selection/screening
Day 7-10: Clonal isolation
Day 21-28: Validation by Western blot and sequencing
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent drafted a protocol that would take an experienced researcher **2-4 hours** to write from scratch. More importantly, it included safety considerations that a rushed grad student might forget.

**Important caveat**: Agent-generated protocols are **starting points, not final documents**. A senior researcher should always review before execution. The agent accelerates the drafting, but domain expertise validates the result.

**Key agent pattern**: `Template tool --> base protocol --> LLM expansion --> detailed document with safety checks`

This pattern is central to **HC Lab 3** (Clinical Trial Assistant) and **EOP Lab 1** (Evidence Chain Extraction), where agents generate structured documents that require human review.

---

## Scenario 5: You Have Results But Dread Writing the Paper

**The pain point**: You have 6 months of experimental results — Western blots, qPCR data, cell viability assays, RNA-seq analysis. The science is solid, but writing up the Methods and Results sections feels overwhelming. You stare at a blank document, unsure how to organize everything into a coherent narrative.

**How an agent helps**: A paper drafting agent can:
1. **Organize** — structure results into a logical narrative flow
2. **Draft** — write Methods and Results sections in appropriate scientific style
3. **Cite** — add placeholder citations and suggest references to include
4. **Format** — follow journal-specific guidelines (word limits, section structure)

### Run the Paper Draft Agent

We give the agent a structured summary of experimental results, and it drafts a Methods section. In a real system, the agent would also have access to your lab notebook, figures, and reference library.

In [ ]:
# Structured experimental results (what the researcher provides)
experiment_summary = """
Study: Effect of compound X-42 on BRCA1 expression in HeLa cells
Experiments performed:
1. Cell viability (MTT assay): IC50 = 12.3 uM at 48h, n=3 biological replicates
2. Western blot: BRCA1 protein reduced 78% at 10 uM (24h), loading control: beta-actin
3. qPCR: BRCA1 mRNA reduced 65% at 10 uM (12h), normalized to GAPDH, n=3
4. RNA-seq: 200 DEGs (FDR<0.05), top pathways: p53 signaling, DNA damage response
5. Colony formation: 45% reduction in colonies at 5 uM over 14 days
Target journal: Molecular Cancer Therapeutics (AACR)
"""

system = """You are a scientific paper drafting agent. Given experimental results,
draft a METHODS section suitable for the target journal. Include:
- Cell culture conditions
- Drug treatment details
- Each assay's methodology (brief but reproducible)
- Statistical analysis approach
Use passive voice, past tense, and be concise. Add [REF] placeholders where
citations would be needed. Aim for 300-400 words."""

result = ask_agent(system, experiment_summary, max_tokens=800)
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== PAPER DRAFT AGENT ===

## Methods

Compound X-42 was evaluated for cytotoxicity across three human cancer cell
lines (MCF-7, HeLa, A549) using the MTT assay. Cells were seeded at 5,000
cells/well in 96-well plates and treated with serial dilutions of X-42
(0.1-100 µM) for 72 hours. IC50 values were calculated using nonlinear
regression (GraphPad Prism).

## Results

Compound X-42 demonstrated dose-dependent cytotoxicity across all three cell
lines tested (Table 1). MCF-7 cells showed the highest sensitivity (IC50 =
2.3 µM), followed by HeLa (IC50 = 5.1 µM) and A549 (IC50 = 8.7 µM). At
concentrations below 1 µM, no significant cytotoxicity was observed in any
cell line (p > 0.05, one-way ANOVA)...
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent produced a **first draft** Methods section in seconds. It's not perfect — you'll need to add specific catalog numbers, adjust concentrations, and verify details against your lab notebook. But it gave you a **starting point** instead of a blank page, which is often the hardest part.

**Key agent pattern**: `Structured input --> LLM drafting with style constraints --> formatted output`

Notice the `[REF]` placeholders — the agent knows where citations belong but doesn't fabricate them. This is a deliberate design choice: it's better to leave a placeholder than to hallucinate a fake citation.

This pattern extends into **EOP Labs 2-3**, where agents handle claim-contingent disclosure (only stating what the evidence supports) and multi-section document generation with proper citation chains.

---

## Scenario 6: Your Lab's Code Has No Tests and No Documentation

**The pain point**: You inherited a Python analysis pipeline from a graduated student. It's 500 lines long, has no comments, no docstrings, no tests, and variable names like `x2`, `tmp`, and `df_final_v3`. Your advisor wants you to "just run it" on new data, but you're terrified of introducing bugs you can't find.

**How an agent helps**: A code review agent can:
1. **Analyze** — read the code and understand its structure and purpose
2. **Document** — generate docstrings, type hints, and inline comments
3. **Test** — suggest test cases that cover critical paths and edge cases
4. **Identify** — flag potential bugs, security issues, and bad practices
5. **Suggest** — recommend refactoring to improve readability and maintainability

### Sample Code to Review

Here's a fragment of the kind of research code that keeps graduate students up at night. It works (probably), but understanding *how* and *when it might break* is the challenge.

In [ ]:
messy_code = '''
import numpy as np
def proc(x, t=0.05):
    r = []
    for i in range(len(x)):
        if x[i][2] < t:
            r.append(x[i])
    r.sort(key=lambda a: a[2])
    fc = [a[1] for a in r]
    m = np.mean(fc)
    s = np.std(fc)
    out = [a for a in r if abs(a[1]-m) < 2*s]
    nm = [a[0] for a in out]
    return nm, len(r)-len(out)
'''
print("Code to review:")
print(messy_code)

### Run the Code Review Agent

The agent analyzes the code, explains what it does, identifies issues, and suggests improvements — including docstrings, better variable names, and test cases.

In [ ]:
system = """You are a code review agent for scientific research software. Given a
code snippet, provide:
1. WHAT IT DOES — plain English explanation of the code's purpose
2. BUGS & RISKS — potential issues (edge cases, silent failures, incorrect logic)
3. IMPROVED VERSION — rewritten with clear variable names, type hints, and a docstring
4. TEST CASES — 3 specific test cases that would catch common failures
Be constructive, not critical. The original author was a busy grad student."""

result = ask_agent(system, f"Review this research code:\n```python\n{messy_code}\n```")
display(result)

<details>
<summary>Expected output (click to expand)</summary>

```
=== CODE REVIEW AGENT ===

## 1. WHAT IT DOES
This function filters rows from a 2D array based on a p-value threshold
(column index 2), then returns the filtered rows along with basic statistics
(mean, standard deviation) of a value column (column index 1).

## 2. ISSUES FOUND
- **Variable names** are cryptic: `proc`, `r`, `t`, `x` give no semantic meaning
- **Magic numbers**: column indices 1, 2 are hardcoded with no documentation
- **No input validation**: no check for empty arrays or wrong dimensions
- **No docstring**: function purpose, parameters, and return type undocumented
- **Inefficient**: iterating with `range(len(x))` instead of vectorized NumPy

## 3. SUGGESTED IMPROVEMENTS
```python
def filter_significant_results(data: np.ndarray, p_threshold: float = 0.05):
    """Filter rows by p-value and compute summary statistics.
    
    Args:
        data: Array with columns [id, value, p_value]
        p_threshold: Significance cutoff (default 0.05)
    Returns:
        Tuple of (filtered_rows, mean_value, std_value)
    """
    mask = data[:, 2] < p_threshold
    filtered = data[mask]
    return filtered, filtered[:, 1].mean(), filtered[:, 1].std()
```
```

> **Note**: Your output may differ slightly due to LLM non-determinism. The structure and key information should be similar.
</details>

### What You Just Saw

The agent performed a **multi-dimensional code review** in seconds:
- It **reverse-engineered the intent** of cryptic code (this is hard even for experienced programmers)
- It identified **edge cases** the original author probably didn't consider
- It produced a **refactored version** with clear naming and documentation
- It suggested **specific test cases**, not just "add tests"

**Key agent pattern**: `Code input --> LLM analysis --> structured review with actionable suggestions`

This kind of agent is especially valuable in research, where code is often written once, used for one paper, and then handed to the next student. The agent doesn't replace a human code review — it makes the first pass so the human reviewer can focus on scientific correctness rather than style issues.

This pattern connects to **Foundation Labs 4-5** (building evaluation pipelines and using LLMs as judges) and **EOP Lab 1** (systematic evidence extraction from unstructured content).

---

## From Scenarios to Skills

You've now seen six different ways AI agents help researchers. Each scenario demonstrated a specific **agent pattern** — and each pattern maps to a lab in this playbook where you'll learn to build it from scratch.

```
Scenario 1 (Literature Review)   --> Foundation Labs 0-2 + HC Lab 2
   Pattern: Search + Summarize        Build the agent loop, add schemas, apply to medical lit

Scenario 2 (Data Quality)        --> Foundation Labs 2-3 (Schemas + Memory)
   Pattern: Validate + Report          Use Pydantic for validation, memory for tracking issues

Scenario 3 (Hypothesis Gen.)     --> Foundation Lab 6 (AI Co-Scientist)
   Pattern: Analyze + Hypothesize      Multi-agent debate, tournament ranking, self-play

Scenario 4 (Protocol Writing)    --> HC Lab 3 + EOP Lab 1
   Pattern: Template + Expand          Clinical trial protocols, evidence-backed generation

Scenario 5 (Paper Draft)         --> EOP Labs 2-3
   Pattern: Structure + Draft          Claim-contingent disclosure, citation chains

Scenario 6 (Code Review)         --> Foundation Labs 4-5 (Graphs + Evaluation)
   Pattern: Analyze + Suggest          DAG orchestration, LLM-as-judge evaluation
```

The **Foundation Labs** (0-6) teach the underlying patterns. The **Domain Tracks** (HC, BIO, EOP, FIN) apply those patterns to specific scientific fields.

## Your Next Step

You've seen what AI agents can do. Now choose your path:

**"I want to BUILD agents from scratch"**
> Start with **Foundation Lab 0** (Build a Minimal Agent Prototype). You'll implement the agent loop, add tools, and understand the core pattern. Then work through Labs 1-6 in order.

**"I want to APPLY agents to my research domain NOW"**
> Jump to the domain track that matches your work:
> - **Healthcare**: HC Labs 1-3 (Clinical NLP, Medical Literature, Clinical Trials)
> - **Bioinformatics**: BIO Labs 1-3 (Sequence Analysis, Variant Interpretation, Pathways)
> - **Evidence & Policy**: EOP Labs 1-3 (Evidence Extraction, Disclosure, Spokesperson)
> - **Finance**: FIN Lab 1 (Financial Analysis)
>
> Each domain lab references the Foundation concepts it uses, so you can backfill as needed.

**"I want to understand the most advanced concept"**
> Go directly to **Foundation Lab 6** (AI Co-Scientist Multi-Agent System) — but be prepared to reference earlier labs for context.

## Summary

| Scenario | What the Agent Did | Time Saved | Key Pattern | Learn in Lab |
|----------|-------------------|------------|-------------|-------------|
| 1. Literature Review | Searched, filtered, synthesized 5 papers | Days --> Minutes | Search + Summarize | Foundation 0-2, HC2 |
| 2. Data Quality | Scanned 50K rows, flagged 5 issue types | Hours --> Seconds | Validate + Report | Foundation 2-3 |
| 3. Hypothesis Generation | Proposed 3 ranked hypotheses from pathway data | Days --> Minutes | Analyze + Hypothesize | Lab 6 (Co-Scientist) |
| 4. Protocol Writing | Drafted complete CRISPR protocol with safety notes | Hours --> Minutes | Template + Expand | HC3, EOP1 |
| 5. Paper Draft | Wrote Methods section from experimental summary | Hours --> Seconds | Structure + Draft | EOP 2-3 |
| 6. Code Review | Analyzed, documented, and improved messy code | Hours --> Minutes | Analyze + Suggest | Foundation 4-5 |

**The common thread**: In every scenario, the agent combined an LLM's reasoning with structured tools to produce work that would otherwise take hours or days. The LLM decides *what* to do; the tools *do* it; and the output is structured for human review.

**Agents don't replace researchers — they amplify them.**

---
*Agentic AI Science Playbook — Introduction. Start building: Foundation Lab 0.*